# Performance Analytics for Mutual Funds

This notebook computes fund-level risk and return metrics, alpha/beta regression against NIFTY100, and a benchmark comparison chart for the top funds.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8')
PROCESSED_DIR = Path('data/processed')
NAV = pd.read_csv(PROCESSED_DIR / '02_nav_history.csv', parse_dates=['date'])
PERF = pd.read_csv(PROCESSED_DIR / '07_scheme_performance.csv')
BENCH = pd.read_csv(PROCESSED_DIR / '10_benchmark_indices.csv', parse_dates=['date'])
NAV = NAV.sort_values(['amfi_code', 'date']).reset_index(drop=True)
NAV['daily_return'] = NAV.groupby('amfi_code')['nav'].pct_change()
BENCH = BENCH.sort_values(['index_name', 'date']).reset_index(drop=True)
BENCH['daily_return'] = BENCH.groupby('index_name')['close_value'].pct_change()
metrics = pd.read_csv('fund_scorecard.csv')
alpha_beta = pd.read_csv('alpha_beta.csv')
print('Loaded outputs with', len(metrics), 'funds and', len(alpha_beta), 'alpha/beta rows.')

## Daily Returns Distribution

Daily fund returns are computed for all 40 schemes. The distribution is validated visually and statistically for reasonableness.

In [ ]:
import plotly.express as px
returns = NAV.dropna(subset=['daily_return'])
fig = px.histogram(returns, x='daily_return', nbins=100, title='Distribution of Daily NAV Returns for All Schemes', labels={'daily_return':'Daily Return'}, height=600, width=900)
fig.update_layout(yaxis_title='Count')
fig.write_image('figures/daily_return_distribution.png')
fig.show()

## CAGR Comparison Table

CAGR is computed for 1-year, 3-year, and 5-year windows for each fund based on available NAV history.

In [ ]:
metrics = pd.read_csv('fund_scorecard.csv')
metrics[['scheme_name','cagr_1yr','cagr_3yr','cagr_5yr']].head(10)

## Sharpe and Sortino Ratios

Risk-adjusted return measures are computed using annualized Sharpe and Sortino ratios with RBI repo proxy Rf=6.5%.

In [ ]:
metrics[['scheme_name','sharpe_ratio','sortino_ratio']].sort_values('sharpe_ratio', ascending=False).head(10)

## Alpha and Beta vs NIFTY100

Alpha and beta are estimated via OLS regression of fund returns on NIFTY100 returns, annualizing alpha.

In [ ]:
alpha_beta[['scheme_name','alpha','beta']].sort_values('alpha', ascending=False).head(10)

## Maximum Drawdown

Worst observed drawdown and the corresponding peak-to-trough date range for each fund.

In [ ]:
metrics[['scheme_name','max_drawdown','max_dd_start_date','max_dd_end_date']].sort_values('max_drawdown').head(10)

## Fund Scorecard

A composite 0-100 score ranks funds using 3-year return, Sharpe, alpha, expense ratio, and drawdown.

In [ ]:
metrics[['scheme_name','score','score_rank']].head(10)

## Benchmark Comparison Chart

The chart below compares the top 5 funds by score against NIFTY50 and NIFTY100 over the last 3 years.

In [ ]:
from PIL import Image
img = Image.open('figures/benchmark_comparison.png')
display(img)